In [1]:
from burau_package.classes.generators import Generators
from burau_package.classes.laurent_matrix import LaurentMatrix
from burau_package.classes.laurent_polynomial import LaurentPolynomial
from burau_package.scripts.free_scripts import reservoir_sampling, word_to_matrix, extend_from_list, extend_garside, projlen
import random

In [2]:
graph = {}
import itertools
from collections import deque

n = 4  # Change this to 3, 5, etc.
def swap(t, i, j):
    lst = list(t)          # 1. Convert to mutable list
    lst[i], lst[j] = lst[j], lst[i]  # 2. Swap elements
    return tuple(lst)      # 3. Convert back to immutable tuple

# Generate all permutations of the list [0, 1, 2, ..., n-1]
# These represent the mapping of indices 0->p[0], 1->p[1], etc.
perms = list(itertools.permutations(range(n)))
visited = [perms[0]]
for perm in perms:
    connected = {}
    for i in range(n-1):
        swapped = swap(perm,i,i+1)
        if not swapped in visited:
            connected[i] = swapped
    visited.append(perm)
    graph[perm] = connected
print(graph)

def get_inverse(p):
    """Calculates the inverse permutation"""
    n = len(p)
    inv = [0] * n
    for i, val in enumerate(p):
        inv[val] = i
    return tuple(inv)

def right_des(perm):
    return list([0,1,2]-graph[perm].keys())

def left_des(perm):
    return right_des(get_inverse(perm))



{(0, 1, 2, 3): {0: (1, 0, 2, 3), 1: (0, 2, 1, 3), 2: (0, 1, 3, 2)}, (0, 1, 3, 2): {0: (1, 0, 3, 2), 1: (0, 3, 1, 2)}, (0, 2, 1, 3): {0: (2, 0, 1, 3), 2: (0, 2, 3, 1)}, (0, 2, 3, 1): {0: (2, 0, 3, 1), 1: (0, 3, 2, 1)}, (0, 3, 1, 2): {0: (3, 0, 1, 2), 2: (0, 3, 2, 1)}, (0, 3, 2, 1): {0: (3, 0, 2, 1)}, (1, 0, 2, 3): {1: (1, 2, 0, 3), 2: (1, 0, 3, 2)}, (1, 0, 3, 2): {1: (1, 3, 0, 2)}, (1, 2, 0, 3): {0: (2, 1, 0, 3), 2: (1, 2, 3, 0)}, (1, 2, 3, 0): {0: (2, 1, 3, 0), 1: (1, 3, 2, 0)}, (1, 3, 0, 2): {0: (3, 1, 0, 2), 2: (1, 3, 2, 0)}, (1, 3, 2, 0): {0: (3, 1, 2, 0)}, (2, 0, 1, 3): {1: (2, 1, 0, 3), 2: (2, 0, 3, 1)}, (2, 0, 3, 1): {1: (2, 3, 0, 1)}, (2, 1, 0, 3): {2: (2, 1, 3, 0)}, (2, 1, 3, 0): {1: (2, 3, 1, 0)}, (2, 3, 0, 1): {0: (3, 2, 0, 1), 2: (2, 3, 1, 0)}, (2, 3, 1, 0): {0: (3, 2, 1, 0)}, (3, 0, 1, 2): {1: (3, 1, 0, 2), 2: (3, 0, 2, 1)}, (3, 0, 2, 1): {1: (3, 2, 0, 1)}, (3, 1, 0, 2): {2: (3, 1, 2, 0)}, (3, 1, 2, 0): {1: (3, 2, 1, 0)}, (3, 2, 0, 1): {2: (3, 2, 1, 0)}, (3, 2, 1, 0): {}}


In [3]:
list_perm = {}
for perm in perms:
    list_perm[perm] = [left_des(perm),right_des(perm)]
    
print(list_perm)



{(0, 1, 2, 3): [[], []], (0, 1, 3, 2): [[2], [2]], (0, 2, 1, 3): [[1], [1]], (0, 2, 3, 1): [[1], [2]], (0, 3, 1, 2): [[2], [1]], (0, 3, 2, 1): [[1, 2], [1, 2]], (1, 0, 2, 3): [[0], [0]], (1, 0, 3, 2): [[0, 2], [0, 2]], (1, 2, 0, 3): [[0], [1]], (1, 2, 3, 0): [[0], [2]], (1, 3, 0, 2): [[0, 2], [1]], (1, 3, 2, 0): [[0, 2], [1, 2]], (2, 0, 1, 3): [[1], [0]], (2, 0, 3, 1): [[1], [0, 2]], (2, 1, 0, 3): [[0, 1], [0, 1]], (2, 1, 3, 0): [[0, 1], [0, 2]], (2, 3, 0, 1): [[1], [1]], (2, 3, 1, 0): [[0, 1], [1, 2]], (3, 0, 1, 2): [[2], [0]], (3, 0, 2, 1): [[1, 2], [0, 2]], (3, 1, 0, 2): [[0, 2], [0, 1]], (3, 1, 2, 0): [[0, 2], [0, 2]], (3, 2, 0, 1): [[1, 2], [0, 1]], (3, 2, 1, 0): [[0, 1, 2], [0, 1, 2]]}


In [4]:
final_perm = {}
for perm in perms:
    allowed_next = []
    for key,value in list_perm.items():
        r_w = list_perm[perm][1]
        l_w1 = value[0]
        if set(l_w1).issubset(set(r_w)) and (key != (0,1,2,3)):
            allowed_next.append(key)
    final_perm[perm] = allowed_next
final_perm.pop((3,2,1,0))    

[(0, 1, 3, 2),
 (0, 2, 1, 3),
 (0, 2, 3, 1),
 (0, 3, 1, 2),
 (0, 3, 2, 1),
 (1, 0, 2, 3),
 (1, 0, 3, 2),
 (1, 2, 0, 3),
 (1, 2, 3, 0),
 (1, 3, 0, 2),
 (1, 3, 2, 0),
 (2, 0, 1, 3),
 (2, 0, 3, 1),
 (2, 1, 0, 3),
 (2, 1, 3, 0),
 (2, 3, 0, 1),
 (2, 3, 1, 0),
 (3, 0, 1, 2),
 (3, 0, 2, 1),
 (3, 1, 0, 2),
 (3, 1, 2, 0),
 (3, 2, 0, 1),
 (3, 2, 1, 0)]

In [5]:
print(final_perm)

{(0, 1, 2, 3): [], (0, 1, 3, 2): [(0, 1, 3, 2), (0, 3, 1, 2), (3, 0, 1, 2)], (0, 2, 1, 3): [(0, 2, 1, 3), (0, 2, 3, 1), (2, 0, 1, 3), (2, 0, 3, 1), (2, 3, 0, 1)], (0, 2, 3, 1): [(0, 1, 3, 2), (0, 3, 1, 2), (3, 0, 1, 2)], (0, 3, 1, 2): [(0, 2, 1, 3), (0, 2, 3, 1), (2, 0, 1, 3), (2, 0, 3, 1), (2, 3, 0, 1)], (0, 3, 2, 1): [(0, 1, 3, 2), (0, 2, 1, 3), (0, 2, 3, 1), (0, 3, 1, 2), (0, 3, 2, 1), (2, 0, 1, 3), (2, 0, 3, 1), (2, 3, 0, 1), (3, 0, 1, 2), (3, 0, 2, 1), (3, 2, 0, 1)], (1, 0, 2, 3): [(1, 0, 2, 3), (1, 2, 0, 3), (1, 2, 3, 0)], (1, 0, 3, 2): [(0, 1, 3, 2), (0, 3, 1, 2), (1, 0, 2, 3), (1, 0, 3, 2), (1, 2, 0, 3), (1, 2, 3, 0), (1, 3, 0, 2), (1, 3, 2, 0), (3, 0, 1, 2), (3, 1, 0, 2), (3, 1, 2, 0)], (1, 2, 0, 3): [(0, 2, 1, 3), (0, 2, 3, 1), (2, 0, 1, 3), (2, 0, 3, 1), (2, 3, 0, 1)], (1, 2, 3, 0): [(0, 1, 3, 2), (0, 3, 1, 2), (3, 0, 1, 2)], (1, 3, 0, 2): [(0, 2, 1, 3), (0, 2, 3, 1), (2, 0, 1, 3), (2, 0, 3, 1), (2, 3, 0, 1)], (1, 3, 2, 0): [(0, 1, 3, 2), (0, 2, 1, 3), (0, 2, 3, 1), (0, 3, 1

In [6]:
mod = None
s_0 = LaurentMatrix([[LaurentPolynomial([0, 0, -1], 0, mod), LaurentPolynomial([0,-1], 0, mod), 0],
                [0, 1, 0],
                [0, 0, 1]])
s_1 = LaurentMatrix([[1, 0, 0],
                [LaurentPolynomial([0,-1], 0, mod), LaurentPolynomial([0, 0, -1], 0, mod), LaurentPolynomial([0,-1], 0, mod)],
                [0, 0, 1]])
s_2 = LaurentMatrix([[1, 0, 0],
                [0, 1, 0],
                [0, LaurentPolynomial([0,-1], 0, mod), LaurentPolynomial([0, 0, -1], 0, mod)]])
arr_mat = [s_0,s_1,s_2]

In [7]:

graph = {}

perms = list(itertools.permutations(range(n)))
visited = [perms[0]]
representation = {}
representation[perms[0]] = tuple()
for perm in perms:
    connected = {}
    for i in range(n-1):
        swapped = swap(perm,i,i+1)
        if not swapped in visited:
            representation[swapped] = tuple(list(representation[perm]) + [i])
            connected[i] = swapped
    visited.append(perm)
    graph[perm] = connected
print(representation)

{(0, 1, 2, 3): (), (1, 0, 2, 3): (0,), (0, 2, 1, 3): (1,), (0, 1, 3, 2): (2,), (1, 0, 3, 2): (0, 2), (0, 3, 1, 2): (2, 1), (2, 0, 1, 3): (1, 0), (0, 2, 3, 1): (1, 2), (2, 0, 3, 1): (1, 0, 2), (0, 3, 2, 1): (2, 1, 2), (3, 0, 1, 2): (2, 1, 0), (3, 0, 2, 1): (2, 1, 0, 2), (1, 2, 0, 3): (0, 1), (1, 3, 0, 2): (0, 2, 1), (2, 1, 0, 3): (1, 0, 1), (1, 2, 3, 0): (0, 1, 2), (2, 1, 3, 0): (1, 0, 1, 2), (1, 3, 2, 0): (0, 2, 1, 2), (3, 1, 0, 2): (2, 1, 0, 1), (3, 1, 2, 0): (2, 1, 0, 1, 2), (2, 3, 0, 1): (1, 0, 2, 1), (2, 3, 1, 0): (1, 0, 2, 1, 2), (3, 2, 0, 1): (2, 1, 0, 2, 1), (3, 2, 1, 0): (2, 1, 0, 2, 1, 2)}


In [8]:
matrix_for_permutation = {}
for _,value in representation.items():
    matrix = LaurentMatrix([[1,0,0],[0,1,0],[0,0,1]],mod)
    for i in value:
        matrix = matrix*arr_mat[i]
    matrix_for_permutation[tuple(value)] = matrix

In [9]:
final_rep_list = {}
for key, value in final_perm.items():
    new_perms = []
    for j in value:
        new_perms.append(representation[j])
    final_rep_list[representation[key]] = new_perms
print(final_rep_list)

{(): [], (2,): [(2,), (2, 1), (2, 1, 0)], (1,): [(1,), (1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1)], (1, 2): [(2,), (2, 1), (2, 1, 0)], (2, 1): [(1,), (1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1)], (2, 1, 2): [(2,), (1,), (1, 2), (2, 1), (2, 1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1), (2, 1, 0), (2, 1, 0, 2), (2, 1, 0, 2, 1)], (0,): [(0,), (0, 1), (0, 1, 2)], (0, 2): [(2,), (2, 1), (0,), (0, 2), (0, 1), (0, 1, 2), (0, 2, 1), (0, 2, 1, 2), (2, 1, 0), (2, 1, 0, 1), (2, 1, 0, 1, 2)], (0, 1): [(1,), (1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1)], (0, 1, 2): [(2,), (2, 1), (2, 1, 0)], (0, 2, 1): [(1,), (1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1)], (0, 2, 1, 2): [(2,), (1,), (1, 2), (2, 1), (2, 1, 2), (1, 0), (1, 0, 2), (1, 0, 2, 1), (2, 1, 0), (2, 1, 0, 2), (2, 1, 0, 2, 1)], (1, 0): [(0,), (0, 1), (0, 1, 2)], (1, 0, 2): [(2,), (2, 1), (0,), (0, 2), (0, 1), (0, 1, 2), (0, 2, 1), (0, 2, 1, 2), (2, 1, 0), (2, 1, 0, 1), (2, 1, 0, 1, 2)], (1, 0, 1): [(1,), (1, 2), (0,), (0, 1), (0, 1, 2), (1, 0), (1, 0, 2), (1, 0, 1),

In [10]:
matrix_for_permutation.pop(tuple())
matrix_for_permutation.pop((2,1,0,2,1,2))

In [12]:
mod = 2
dict = matrix_for_permutation
base_dict = [ ]
for key, value in matrix_for_permutation.items():
    base_dict.append([tuple([key]),value])
reservoir_sampling(dict=dict,transition_dict= final_rep_list, results = base_dict, invariant = projlen, extending_function=extend_garside, verbose=1)

3
2 {4: 5, 5: 23, 6: 55, 7: 49, 8: 27}
5
3 {6: 6, 7: 39, 8: 152, 9: 234, 10: 302, 11: 163, 12: 60}
7
4 {8: 7, 9: 56, 10: 314, 11: 680, 12: 1343, 13: 1282, 14: 1168, 15: 430, 16: 125}
9
5 {10: 8, 11: 74, 12: 560, 13: 1555, 14: 4152, 15: 5598, 16: 7646, 17: 5275, 18: 3740, 19: 1010, 20: 254}
11
6 {12: 9, 13: 93, 14: 909, 15: 3070, 16: 10000, 17: 10000, 18: 10000, 19: 10000, 20: 10000, 21: 10000, 22: 10000, 23: 2223, 24: 511}
13
7 {14: 10, 15: 113, 16: 1390, 17: 5453, 18: 10000, 19: 10000, 20: 10000, 21: 10000, 22: 10000, 23: 10000, 24: 10000, 25: 10000, 26: 10000, 27: 4709, 28: 1024}
15
8 {16: 11, 17: 137, 18: 2014, 19: 8854, 20: 10000, 21: 10000, 22: 10000, 23: 10000, 24: 10000, 25: 10000, 26: 10000, 27: 10000, 28: 10000, 29: 10000, 30: 10000, 31: 9748, 32: 2049}
17


KeyboardInterrupt: 